# Session 8 Mini RAG Lab

This notebook gives you a **lightweight retrieval baseline**.

Your goal is **not** to trust the output blindly.
Your goal is to inspect retrieval results, judge source authority, and explain failure modes.


## Step 1 — Load the corpus
Check how many documents and chunks you have.


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

BASE_DIR = Path.cwd().resolve().parent if (Path.cwd()/"dataset").exists() is False else Path.cwd()
CORPUS_DIR = BASE_DIR / 'dataset' / 'corpus'
QUERY_FILE = BASE_DIR / 'dataset' / 'eval_queries.csv'

def load_documents(corpus_dir: Path):
    docs = []
    for path in sorted(corpus_dir.glob('*.md')):
        text = path.read_text(encoding='utf-8')
        doc_id = path.name.split('_')[0]
        docs.append({'doc_id': doc_id, 'filename': path.name, 'text': text})
    return docs

docs = load_documents(CORPUS_DIR)
len(docs), [d['doc_id'] for d in docs]


## Step 2 — Chunk by paragraph
This is the default baseline. Later you can change the chunking strategy.


In [ ]:
def chunk_by_paragraph(documents):
    chunks = []
    for doc in documents:
        paragraphs = [p.strip() for p in doc['text'].split('\n\n') if p.strip()]
        for i, para in enumerate(paragraphs, start=1):
            chunks.append({'chunk_id': f"{doc['doc_id']}_P{i}", 'doc_id': doc['doc_id'], 'text': para})
    return pd.DataFrame(chunks)

chunk_df = chunk_by_paragraph(docs)
chunk_df.head()


## Step 3 — Build a baseline retriever


In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')
matrix = vectorizer.fit_transform(chunk_df['text'])
matrix.shape


## Step 4 — Query and inspect top-3 results
Try Q04, Q05, and Q10 first.


In [ ]:
query_df = pd.read_csv(QUERY_FILE)
query_df[['query_id','query','designed_failure_mode']]


In [ ]:
def retrieve(query, top_k=3):
    q = vectorizer.transform([query])
    scores = cosine_similarity(q, matrix).flatten()
    out = chunk_df.copy()
    out['score'] = scores
    return out.sort_values('score', ascending=False).head(top_k)[['chunk_id','doc_id','score','text']]

for qid in ['Q04','Q05','Q10']:
    row = query_df[query_df['query_id']==qid].iloc[0]
    print('='*90)
    print(qid, row['query'])
    display(retrieve(row['query'], top_k=3))


## Step 5 — Manual judgment
For each query, write down:
- Which retrieved doc is actually authoritative?
- Which retrieved doc is misleading?
- What failure mode do you see?


## Optional Improvement A — Synonym rewriting
Example: replace `uneven left-right fan draw` with `fan current imbalance` before retrieval.


In [ ]:
def rewrite_query(query: str) -> str:
    mapping = {
        'uneven left-right fan draw': 'fan current imbalance',
        'thermal runaway': 'cell venting precursor',
    }
    new_q = query
    for k, v in mapping.items():
        new_q = new_q.replace(k, v)
    return new_q

test_query = query_df[query_df['query_id']=='Q05'].iloc[0]['query']
print('original:', test_query)
print('rewritten:', rewrite_query(test_query))
display(retrieve(rewrite_query(test_query), top_k=3))


## Optional Improvement B — Authority filtering
Idea: remove outdated or informational-only sources before ranking.


In [ ]:
catalog = pd.read_csv(BASE_DIR / 'dataset' / 'corpus_catalog.csv')
catalog


In [ ]:
allowed = set(catalog[~catalog['status'].isin(['outdated_archive','informational_only'])]['doc_id'])
filtered_chunk_df = chunk_df[chunk_df['doc_id'].isin(allowed)].reset_index(drop=True)
filtered_vectorizer = TfidfVectorizer(stop_words='english')
filtered_matrix = filtered_vectorizer.fit_transform(filtered_chunk_df['text'])

def retrieve_filtered(query, top_k=3):
    q = filtered_vectorizer.transform([query])
    scores = cosine_similarity(q, filtered_matrix).flatten()
    out = filtered_chunk_df.copy()
    out['score'] = scores
    return out.sort_values('score', ascending=False).head(top_k)[['chunk_id','doc_id','score','text']]

display(retrieve_filtered(query_df[query_df['query_id']=='Q01'].iloc[0]['query'], top_k=3))


## Final reflection
Before submitting, answer:
1. Where can retrieval look convincing but still be wrong?
2. What one change most improved quality?
3. Why does source authority matter in engineering RAG?
